# Git: undo, history, and working with another person

## 📚 Learning Objectives

By the end of this lesson you will be able to:

- Read a working tree with `git status` and `git diff` and say **exactly** what your next commit will contain, before you make it.
- Stage deliberately — commit one change while leaving another on the floor.
- Write a commit message that answers the question someone will actually ask about it in six months.
- Branch to try something, and abandon the attempt without damaging anything.
- Find the commit that broke a result, with `git log -S`, `git blame` and `git bisect run`.
- Recover from the five mistakes that really happen: wrong edit, wrong file staged, wrong message, wrong commit shipped, wrong `reset`.
- Run the two-person loop end to end: **branch → push → review → merge** — with no network and no GitHub account.

## 🔗 Where this fits

**Builds on:** TOOLING lesson 01 — you need a terminal, and you need to be able to move around a filesystem, before a single command in this notebook makes sense.

**Used later in:** every lesson in this strand that asks you to hand work to another person — the peer-review lesson above all — and the capstone, where you are handed a *repository* rather than a notebook.

**What this is not:** a git tutorial. Git has well over a hundred commands. This lesson teaches about twenty, chosen on one criterion: *these are the ones you reach for when something has gone wrong and other people are waiting.*

---

## 🎯 Four lines, and the hole that five leaked answers hid in

This repository — the one this notebook is sitting inside — has a public, honest history. Some of its commits record defects that were found and fixed. One of them is four lines long.

On 24 August 2026 a check was added to `tools/verify/check_no_answer_keys.py`. Until then the check looked for the *word* "Answer". The new four lines counted **```** fences instead, on the reasoning that an odd number of fences means an opening fence was lost — and a lost opening fence is how a block of solution code stops looking like a code block and starts looking like the page. The commit message says so in one sentence:

> *That asymmetry is how the 5 exposed solutions hid from the marker-based check.*

Eight days later the same idea failed again, at a larger size, and the history records that too: `fix(verify): the leak detector was blind to every code-writing answer - verb list removed`.

You will read both of these commits in a moment, straight out of the repository, with `git show`. That is the point of this lesson. **Version history is not a backup. It is the only written record of why the code is the way it is** — and the only tool that can answer, on a codebase you have never seen, the question every new engineer is asked in their first week:

> *"This number used to be right. When did it stop being right, and what changed?"*

By the end of this notebook you will have answered exactly that question, on a real repository, with `git bisect run`, in four automated checks.

---

## 🧠 One picture: git has three places, not one

Almost every confusing thing about git dissolves once you hold this picture. A file can exist in three states at the same time, with three different contents:

```
   WORKING TREE              INDEX  (the "staging area")            HEAD  (history)
   the files on disk         what your NEXT commit will             what is already
   as you just edited them   contain — chosen by you, one           recorded, and is
                             file or one hunk at a time             very hard to lose

        │                              │                                   │
        │  ───────  git add  ────────► │                                   │
        │ ◄─── git restore --staged ── │                                   │
        │                              │  ────── git commit ─────────────► │
        │ ◄──────────────  git restore <file>  ─────────────────────────── │
```

- `git status` tells you **which of the three disagree**.
- `git diff` shows **working tree vs index** — "what have I changed and not yet staged".
- `git diff --staged` shows **index vs HEAD** — "what exactly am I about to commit".

Those two diffs are different questions, and mixing them up is the single most common source of a commit that contains something the author did not intend to ship.

## The six operations that matter under pressure

| # | Question you are actually asking | Command |
|---|---|---|
| 1 | What have I changed? | `git status`, `git diff`, `git diff --staged` |
| 2 | What exactly am I committing? | `git add <path>`, `git add -p`, `git restore --staged` |
| 3 | Will this message help anyone in six months? | `git commit -m` (and the message itself) |
| 4 | Can I try this without risking main? | `git switch -c`, `git switch -`, `git branch -D` |
| 5 | When did this break, and why? | `git log -S`, `git blame`, `git bisect run` |
| 6 | How do I get out of this? | `git restore`, `git revert`, `git reflog` |

Then a seventh, which is the whole reason the other six exist: **hand the work to another person and let them find what you missed.**

---

In [1]:
# WHAT: build a throwaway "laboratory" directory somewhere in the system temp area, and find
#       this repository's own root so we can READ its real history later.
# WHY:  every git command in this notebook that WRITES runs inside the laboratory. Nothing in
#       this notebook can modify the repository you are reading it from — the only commands
#       pointed at this repository are read-only ones (log, show, blame).
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

LAB = tempfile.mkdtemp(prefix="git-lab-")          # e.g. /var/folders/.../git-lab-xxxx
os.environ["LAB"] = LAB                            # every %%bash cell below reads $LAB
os.environ["PY"] = sys.executable                  # the exact python running this kernel
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"        # keeps __pycache__ out of the demo repo
os.environ["GIT_PAGER"] = "cat"                    # never open a pager inside a notebook

# Walk upwards from this notebook until we find the directory that holds .git — that is the
# root of THIS repository. On a ZIP download there is no .git and the read-only cells skip.
here = Path.cwd().resolve()
repo = next((p for p in [here, *here.parents] if (p / ".git").exists()), None)
os.environ["REPO"] = str(repo) if repo else ""

# Four real commits from this repository are quoted later. Check they are actually reachable
# (a shallow "--depth 1" clone would not have them) so the cells can skip honestly instead
# of printing an error and pretending it is a lesson.
COMMITS = ["a1249287", "79309823", "5113c385", "ced3fad6"]
have = False
if repo:
    have = all(
        subprocess.run(["git", "-C", str(repo), "cat-file", "-e", c + "^{commit}"],
                       capture_output=True).returncode == 0
        for c in COMMITS
    )
os.environ["HAVE_HISTORY"] = "1" if have else "0"

print("git version   :", subprocess.run(["git", "--version"], capture_output=True,
                                        text=True).stdout.strip())
print("laboratory    :", LAB)
print("this repository:", repo if repo else "(not a git checkout — read-only cells will skip)")
print("its history is readable:", have)

git version   : git version 2.50.1 (Apple Git-155)
laboratory    : /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/git-lab-c955n6hk
this repository: /Users/abdullah/Downloads/AI Diploma
its history is readable: True


In [2]:
%%bash
# WHAT: build the small project you will investigate for the rest of this lesson: twelve
#       commits, made the ordinary way, one `git add` and one `git commit` at a time.
# WHY:  a real repository's history is not generated, it accumulates. One of these twelve
#       commits silently breaks a reported number. You are not told which one.
set -uo pipefail
cd "${LAB:?run the setup cell first}" || exit 1

mkdir -p analysis && cd analysis || exit 1
git init -q -b main                       # -b main names the first branch explicitly
git config user.name  "Student"           # set LOCALLY, so this repo cannot inherit or
git config user.email "student@example.com"   # disturb your own global git identity
git config commit.gpgsign false

snap () { git add -A && git commit -q -m "$1"; }   # tiny helper: stage everything, commit

printf '__pycache__/\n*.pyc\n' > .gitignore        # 01

cat > metrics.py <<'EOF'
"""Metrics computed from a 2x2 confusion matrix laid out as [[TN, FP], [FN, TP]]."""


def accuracy(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    return (tn + tp) / (tn + fp + fn + tp)
EOF
snap "feat(metrics): accuracy from a 2x2 confusion matrix

Layout is [[TN, FP], [FN, TP]] so results can be checked straight against
sklearn.metrics.confusion_matrix without transposing anything."

cat > matrices.py <<'EOF'
"""Two confusion matrices for the same fraud model, over the same 3200 transactions.

BASELINE is the model fitted with default settings.
BALANCED is the same model refitted with class_weight='balanced'.
"""

BASELINE = [[3191, 3], [3, 3]]
BALANCED = [[3176, 18], [3, 3]]
EOF
snap "data(metrics): the two confusion matrices this project reports on"                   # 02

cat >> metrics.py <<'EOF'


def recall(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    return tp / (tp + fn)
EOF
snap "feat(metrics): recall - the fraction of real fraud we actually caught"               # 03
git rev-parse --short HEAD > "$LAB/GOOD"   # remember this one: recall was correct here

cat > README.md <<'EOF'
# Fraud metrics

Accuracy on the baseline model is 99.81%. A model that answers "not fraud" to
every single transaction scores the same 99.81%. Read recall instead.
EOF
snap "docs: say plainly that accuracy is useless on this data"                             # 04

cat > report.py <<'EOF'
"""Print one line per model so the two can be compared at a glance."""

import matrices
import metrics

for name in ("BASELINE", "BALANCED"):
    cm = getattr(matrices, name)
    n = sum(cm[0]) + sum(cm[1])
    print(f"{name:<9} n={n}  accuracy={metrics.accuracy(cm):.2f}  recall={metrics.recall(cm):.2f}")
EOF
snap "feat(report): print a one-line summary for both models"                              # 05

cat > metrics.py <<'EOF'
"""Metrics computed from a 2x2 confusion matrix laid out as [[TN, FP], [FN, TP]]."""


def _unpack(cm):
    """Pull the four cells out once so every metric reads the same way."""
    tn, fn = cm[0]
    fp, tp = cm[1]
    return tn, fp, fn, tp


def accuracy(cm):
    tn, fp, fn, tp = _unpack(cm)
    return (tn + tp) / (tn + fp + fn + tp)


def recall(cm):
    tn, fp, fn, tp = _unpack(cm)
    return tp / (tp + fn)
EOF
snap "refactor(metrics): unpack the matrix once instead of in every function"              # 06

cat >> metrics.py <<'EOF'


def precision(cm):
    tn, fp, fn, tp = _unpack(cm)
    return tp / (tp + fp)
EOF
snap "feat(metrics): precision, so the false-alarm cost is visible"                        # 07

cat > report.py <<'EOF'
"""Print one line per model so the two can be compared at a glance."""

import matrices
import metrics

for name in ("BASELINE", "BALANCED"):
    cm = getattr(matrices, name)
    n = sum(cm[0]) + sum(cm[1])
    print(f"{name:<9} n={n}  accuracy={metrics.accuracy(cm):.2f}"
          f"  recall={metrics.recall(cm):.2f}  precision={metrics.precision(cm):.2f}")
EOF
snap "feat(report): show precision next to recall"                                         # 08

cat >> README.md <<'EOF'

class_weight='balanced' moves false positives from 3 to 18 and leaves recall
exactly where it was.
EOF
snap "docs: record what class_weight='balanced' actually changed"                          # 09

cat > report.py <<'EOF'
"""Print one line per model so the two can be compared at a glance."""

import matrices
import metrics

for name in ("BASELINE", "BALANCED"):
    cm  = getattr(matrices, name)
    n   = sum(cm[0]) + sum(cm[1])
    print(f"{name:<9} n={n}  accuracy={metrics.accuracy(cm):.2f}"
          f"  recall={metrics.recall(cm):.2f}  precision={metrics.precision(cm):.2f}")
EOF
snap "style(report): align the assignments"                                                # 10

cat > report.py <<'EOF'
"""Print one line per model so the two can be compared at a glance."""

import matrices
import metrics

for name in ("BASELINE", "BALANCED"):
    cm  = getattr(matrices, name)
    n   = sum(cm[0]) + sum(cm[1])
    print(f"{name:<9} n={n}  accuracy={metrics.accuracy(cm):.4f}"
          f"  recall={metrics.recall(cm):.4f}  precision={metrics.precision(cm):.4f}")
EOF
snap "feat(report): four decimals - two was hiding the difference between the models"      # 11

cat >> README.md <<'EOF'

Usage: `python report.py`
EOF
snap "docs: add a usage line to the README"                                                # 12

echo "=== the history you have just built ==="
git log --oneline
echo
echo "=== and what the project reports today ==="
"$PY" report.py

=== the history you have just built ===


74d8381 docs: add a usage line to the README


48d21e0 feat(report): four decimals - two was hiding the difference between the models
7f00cba style

(report): align the assignments
9924b60 docs: record what class_weight='balanced' actually changed
d

96daca feat(report): show precision next to recall
58dc01c feat(metrics): precision, so the false-al

...

fraction of real fraud we actually caught
1936e5f data(metrics): the two confusion matrices this pro

ject reports on
07a52e9 feat(metrics): accuracy from a 2x2 confusion matrix

=== and what the projec

t reports today ===


BASELINE  n=3200  accuracy=0.9981  recall=0.5000  precision=0.5000
BALANCED  n=3200  accuracy=0.9934

  recall=0.1429  precision=0.5000


### 👁️ Read that output before going on

Twelve commits, and one printed report. Compare the last two lines of the report with the last two lines of the `README.md` that commit 09 wrote:

> *class_weight='balanced' moves false positives from 3 to 18 and leaves recall exactly where it was.*

The report says `BALANCED` has **recall = 0.1429** and `BASELINE` has **recall = 0.5000**. Recall did not stay where it was; it fell by more than two thirds. Either the README is wrong, or the code is.

Hold that. We come back to it in operation 5 and settle it in four automated checks — **without reading `metrics.py`**.

---

## 1️⃣ Seeing what changed

You have been asked to add a `total()` helper — the number of rows the matrix summarises. While you were in there you also dropped a `print("DEBUG ...")` into `report.py`, the way everyone does.

Two files are now dirty. **`git status` names the files; `git diff` shows the characters.**

In [3]:
%%bash
# WHAT: make two unrelated edits, then look at them the two ways git offers.
# WHY:  `git status` is the map (which files), `git diff` is the territory (which characters).
#       Committing without reading the second one is how debug prints reach production.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

cat >> metrics.py <<'EOF'


def total(cm):
    """How many rows this matrix summarises."""
    return sum(cm[0]) + sum(cm[1])
EOF
printf '\nprint("DEBUG: reached the end of report.py")\n' >> report.py

echo "=== git status ==="
git status
echo
echo "=== git status --short  (the form you will actually use) ==="
git status --short
echo
echo "=== git diff  (working tree vs index: what I changed and have NOT staged) ==="
git diff

=== git status ===


On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be comm

itted)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   metrics.

py
	modified:   report.py

no changes added to commit (use "git add" and/or "git commit -a")



=== git status --short  (the form you will actually use) ===


...

(cm[0]) + sum(cm[1])
diff --git a/report.py b/report.py
index 42b4d49..c4fd6e7 100644
--- a/report.p

y
+++ b/report.py
@@ -8,3 +8,5 @@ for name in ("BASELINE", "BALANCED"):
     n   = sum(cm[0]) + sum(

cm[1])
     print(f"{name:<9} n={n}  accuracy={metrics.accuracy(cm):.4f}"
           f"  recall={met

rics.recall(cm):.4f}  precision={metrics.precision(cm):.4f}")
+
+print("DEBUG: reached the end of re

port.py")


### How to read a diff, once

```
(schematic — not a diff from this project)
diff --git a/some_file.py b/some_file.py   the file
@@ -8,3 +8,8 @@                          @@ old-start,old-length new-start,new-length @@
 def f(x):                                a line with a leading SPACE is context, unchanged
-    return x + 1                         a leading MINUS is a line that was there and is gone
+    return x + 2                         a leading PLUS is a line that is new
```

`git status --short` uses two columns, and the columns are the two left-hand boxes of the picture above: **column 1 is the index, column 2 is the working tree.**

| code | meaning |
|---|---|
| ` M` | modified on disk, **not** staged |
| `M ` | modified and staged, disk matches the index |
| `MM` | staged — and then modified again afterwards |
| `A ` | a new file, staged |
| `??` | git has never heard of this file |
| `UU` | both sides changed it during a merge — a conflict |

---

## 2️⃣ Staging deliberately

The debug print must not ship. The `total()` helper should. They are in different files, so one `git add` separates them — and `git diff --staged` proves the separation *before* you commit rather than after.

When the two changes are in the **same** file, the command is `git add -p` (`-p` for *patch*): git shows you one hunk at a time and asks `Stage this hunk [y,n,q,a,d,s,e,?]?`. It is interactive, so it cannot run inside a notebook — it is the first thing you will do by hand in "Your turn".

In [4]:
%%bash
# WHAT: stage ONE of the two dirty files, then look at both diffs side by side.
# WHY:  `git diff --staged` is the last honest look at what you are about to record. It is
#       the difference between "I think I only changed metrics.py" and knowing it.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

git add metrics.py

echo "=== git status --short ==="
git status --short
echo "    (M in column 1 = metrics.py is staged;  M in column 2 = report.py is not)"
echo
echo "=== git diff --staged  (index vs HEAD: EXACTLY what the next commit contains) ==="
git diff --staged
echo
echo "=== git diff  (what is still on the floor, and will not be committed) ==="
git diff

=== git status --short ===


M  metrics.py
 M report.py


    (M in column 1 = metrics.py is staged;  M in column 2 = report.py is not)

=== git diff --staged

  (index vs HEAD: EXACTLY what the next commit contains) ===


diff --git a/metrics.py b/metrics.py
index 4b002c4..c3542ec 100644
--- a/metrics.py
+++ b/metrics.py

...


=== git diff  (what is still on the floor, and will not be committed) ===


diff --git a/report.py b/report.py
index 42b4d49..c4fd6e7 100644
--- a/report.py
+++ b/report.py
@@ 

-8,3 +8,5 @@ for name in ("BASELINE", "BALANCED"):
     n   = sum(cm[0]) + sum(cm[1])
     print(f"{

name:<9} n={n}  accuracy={metrics.accuracy(cm):.4f}"
           f"  recall={metrics.recall(cm):.4f} 

 precision={metrics.precision(cm):.4f}")
+
+print("DEBUG: reached the end of report.py")


## 3️⃣ A message someone can use in six months

A commit message has one job: to be found and understood by a stranger — often you, later — who is asking *why does this line exist?* Almost every rule you will read about messages follows from that.

**The shape that works:**

```
type(scope): one line, imperative, under ~72 characters, no full stop

Why this change was necessary — the problem, not the patch. The diff already
shows what changed; it can never show what you were thinking.

What you checked, and what it printed. Numbers if you have them.
```

- **Imperative mood** — "fix the unpack order", not "fixed" or "fixes". It reads as an instruction *to the codebase*, and it matches what git itself writes for you (`Merge branch...`, `Revert "..."`).
- **The first line is a headline, not a title.** It appears alone in `git log --oneline`, in `git blame`, in every pull-request list, in `git bisect` output. If it only makes sense next to the diff, it has failed.
- **`type(scope):`** is a convention (`feat`, `fix`, `docs`, `refactor`, `test`, `chore`, `perf`, `style`) that costs nothing and makes `git log --oneline | grep '^.\{8\} fix'` a useful thing to type.

**The test to apply to your own message:** *could a colleague who has never seen this code decide, from the message alone, whether this commit is the one that broke their result?* That is not a stylistic question. In operation 5 you will scan a list of subject lines looking for exactly that, and the messages will either help you or waste your afternoon.

In [5]:
%%bash
# WHAT: commit only the staged file, with a message written to the shape above, then throw
#       the debug print away.
# WHY:  -m twice gives you a subject and a body without opening an editor. The second -m
#       becomes the paragraph under the blank line, which is where the "why" belongs.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

git commit -q -m "feat(metrics): total(), the number of rows a matrix summarises" \
                -m "report.py recomputes this inline in its own loop; both call sites will
move to this one so a matrix with a wrong row count is caught in one place
rather than two."

echo "=== the commit, as a stranger will meet it ==="
git log -1 --stat
echo
echo "=== and the debug print is still sitting on the floor, uncommitted ==="
git status --short
git restore report.py            # operation 6 arriving early: discard an unstaged edit
echo "after git restore report.py:"
git status --short && echo "(nothing printed above: the working tree is clean)"

=== the commit, as a stranger will meet it ===


commit a185b3cc60166f7a21bf8ba4c049ed057749b508
Author: Student <student@example.com>
Date:   Sat Se

p 5 00:51:10 2026 +0300

    feat(metrics): total(), the number of rows a matrix summarises
    
   

 report.py recomputes this inline in its own loop; both call sites will
    move to this one so a ma

trix with a wrong row count is caught in one place
    rather than two.

 metrics.py | 5 +++++
 1 fi

le changed, 5 insertions(+)



=== and the debug print is still sitting on the floor, uncommitted ===


 M report.py


after git restore report.py:


(nothing printed above: the working tree is clean)


### Real messages, from this repository

Style advice is cheap. Below are twelve subject lines from the history of the repository this notebook is sitting inside, one message printed in full, and one subject line whose body deliberately stays off this page. Read them as a stranger would, and ask the test question of each: *could I decide from this alone whether it is the commit that broke my result?*

In [6]:
%%bash
# WHAT: read this repository's OWN history. Every command here is read-only: log and show
#       cannot change a repository.
# WHY:  the messages you are about to copy are the ones written by people who had to live
#       with them afterwards.
set -uo pipefail
if [ -z "${REPO:-}" ] || [ "${HAVE_HISTORY:-0}" != "1" ]; then
  echo "This repository's history is not available here (ZIP download, or a shallow clone)."
  echo "Everything else in this notebook still runs. Skipping."
  exit 0
fi

echo "=== twelve subject lines, the way git log --oneline shows them ==="
git -C "$REPO" log --oneline -12
echo
echo "=== one message in full: a check that was wrong, and why it was wrong ==="
git -C "$REPO" show -s --format='commit %h   %an   %ad%n%n%B' --date=short 79309823
echo "=== and the subject line of a commit whose body we deliberately do not print ==="
git -C "$REPO" log -1 --format='%h  %s' ced3fad6
echo "    (that one's body quotes exam answers, so it stays off a student page.)"

=== twelve subject lines, the way git log --oneline shows them ===


0e7e9e2d fix(verify): parse gate could not read an assignment-form IPython magic


8883c8dd feat(retrieval+tooling): the two changes the evidence ranked first, built from what already

 existed
aca8ab79 fix(c08,c10): 14 lessons gained Discuss and Where-this-breaks - and the audit foun

d four real defects
540b091e chore(hygiene): remove two __pycache__ directories the structure gate f

...

-Authored-By: Claude Fable 5 <noreply@anthropic.com>



=== and the subject line of a commit whose body we deliberately do not print ===


ced3fad6  fix(c04-exam): key and rubric disagreed on five of six items - a correct paper scored 25 o

f 30 wrong


    (that one's body quotes exam answers, so it stays off a student page.)


### 👁️ What those messages do that a bad one cannot

Look at the long one. It states the failure (*the check fired only when a prompt matched a verb list*), it quantifies it (*missed 11 of the 26 Course 03 leaks and found nothing at all in Course 02*), it names what it now does instead (*judges structure rather than wording*), and it records what re-running found. Six months from now, somebody who is wondering whether to trust that check has the whole story without reading one line of Python.

Now compare it with the messages nobody can use: `update`, `fix bug`, `wip`, `changes`, `asdf`. Every one of them costs a future reader a `git show` — and, when you are bisecting through forty commits, forty of them.

---

## 4️⃣ Branching, so trying something is free

A branch in git is not a copy of anything. **It is a movable label pointing at one commit.** Creating one is instant and costs nothing, whatever the size of the repository, and that is why "make a branch" is the correct answer to *"can I try something?"* every single time.

- `git switch -c try-something` — create the label at HEAD, and move onto it.
- `git switch -` — go back to where you were (`-` means "the previous branch", exactly like `cd -`).
- `git branch -D try-something` — delete the label. The commits are still on disk; only the name is gone, and `git reflog` can still reach them for weeks.

The reason this matters for you specifically: on a branch, an experiment that does not work is deleted with one command instead of being un-picked by hand out of a file you have already half-rewritten.

In [7]:
%%bash
# WHAT: branch, do work on the branch, go back, compare the two, then throw the branch away.
# WHY:  to show that main is untouched the whole time - which is the only reason branching
#       makes an experiment cheap.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

git switch -c try-f1
cat >> metrics.py <<'EOF'


def f1(cm):
    p, r = precision(cm), recall(cm)
    return 2 * p * r / (p + r)
EOF
git commit -q -am "feat(metrics): f1, the harmonic mean of precision and recall"

echo "=== branches now (the * marks where HEAD is) ==="
git branch
echo
git switch -                    # back to main, in one keystroke
echo
echo "=== what is on the branch and not on main ==="
git log --oneline main..try-f1
echo
echo "=== is main affected? ==="
if grep -q "def f1" metrics.py; then echo "f1 IS on main"; else echo "f1 does not exist on main at all"; fi
echo
echo "=== abandon the experiment ==="
git branch -D try-f1
git branch
echo "(the commit is still on disk and reachable through git reflog; only the name is gone)"

Switched to a new branch 'try-f1'


=== branches now (the * marks where HEAD is) ===


  main
* try-f1


Switched to branch 'main'


=== what is on the branch and not on main ===


...

f1 does not exist on main at all



=== abandon the experiment ===


Deleted branch try-f1 (was 4da244d).


* main


(the commit is still on disk and reachable through git reflog; only the name is gone)


## 5️⃣ When did this break, and why?

Back to the contradiction from the start: the README says recall did not move, the report says it fell from 0.5000 to 0.1429. Thirteen commits now, counting the `total()` one you just made. Which of them did it?

There are three tools, and they answer three genuinely different questions. Learn all three; you will use `git log -S` most often and `git bisect` when nothing else works.

| tool | the question it answers | when it works |
|---|---|---|
| `git log -S"text"` | *which commit added or removed this exact string?* | you know a string that appears in the broken code |
| `git blame <file>` | *who last touched **this line**, and in which commit?* | you already know which line is wrong |
| `git bisect` | *which commit changed this behaviour?* | you know it is broken **now** and was fine **then** — and nothing else |

`git log -S` is called the **pickaxe**. It does not search commit messages; it searches the *content of the diffs*, which is why it finds the commit that introduced a line even when the message says nothing about it.

In [8]:
%%bash
# WHAT: use the pickaxe on this repository's real history to find the commit that introduced
#       one specific line, then read that commit's diff in full.
# WHY:  this is the fastest route from "a strange line of code" to "the reason it is there".
set -uo pipefail
if [ -z "${REPO:-}" ] || [ "${HAVE_HISTORY:-0}" != "1" ]; then
  echo "This repository's history is not available here. Skipping."
  exit 0
fi

echo '=== git log -S"unbalanced" -- tools/verify/  (which commit introduced that word?) ==='
git -C "$REPO" log --oneline -S"unbalanced" -- tools/verify/
echo
echo "=== and here is that commit, in full: message and diff ==="
git -C "$REPO" show --format='commit %h   %an   %ad%n%n%B' --date=short a1249287

=== git log -S"unbalanced" -- tools/verify/  (which commit introduced that word?) ===


a1249287 feat(verify): answer-key gate now flags unbalanced code fences in quizzes


=== and here is that commit, in full: message and diff ===


commit a1249287   Abdullah Alwabel   2026-08-24

feat(verify): answer-key gate now flags unbalanced 

code fences in quizzes

That asymmetry is how the 5 exposed solutions hid from the marker-based chec

...

r: {m.group(0)[:40]!r}"))
+            # An odd number of ``` fences means one opener was lost (bili

ngual strips ate a few),
+            # which leaves the solution code below a question visible to s

tudents.
+            if txt.count("```") % 2:
+                bad.append((f, "unbalanced code fenc

es — a solution block may be exposed"))
     if bad:
         print(f"ANSWER-KEY GATE: {len(bad)} 

violation(s)")
         for f, why in bad:


### 👁️ Read that diff

The pickaxe returned **one** commit. Its diff is four added lines and nothing else, and those four lines are a complete argument:

```python
+            # An odd number of ``` fences means one opener was lost (bilingual strips ate a few),
+            # which leaves the solution code below a question visible to students.
+            if txt.count("```") % 2:
+                bad.append((f, "unbalanced code fences — a solution block may be exposed"))
```

Two lines of code, two lines of comment saying *why*, and a commit message giving the evidence. That is the standard to aim at — not because it is tidy, but because you have just reconstructed the entire reasoning of a change made by someone else, in one command, without asking them.

Note what made it findable: the string `unbalanced` is **in the diff**, not in the message. `git log --grep` would have missed it.

---

### `git bisect`: when you have no idea where to look

`git blame` and the pickaxe both need a guess. `git bisect` needs none. You tell it one commit where the behaviour was **good** and one where it is **bad**; it checks out the commit halfway between and asks you which it is; and each answer halves what is left. Ten candidate commits need at most four checks; a hundred need seven; a thousand need ten — **⌈log₂ n⌉**. Watch the `Bisecting: N revisions left to test after this` line in the output below count down.

`git bisect run <command>` automates even that: give it any command that exits **0 for good** and **non-zero for bad**, and git drives the whole search itself. Exit code **125** means *"cannot tell — skip this commit"*, which is how you survive a range where the code did not even import.

So the test has to exist first. Ours is a dozen lines, one of which is the assertion, and it lives **outside** the repository — because bisect checks out old commits, and a test file committed only at the tip would vanish underneath itself.

In [9]:
%%bash
# WHAT: write a one-assertion test outside the repo, then let git bisect run drive the search
#       for the commit that changed recall(BALANCED).
# WHY:  this is the whole technique. Turning "it is wrong" into a command that exits 0 or 1 is
#       the only hard part; git does the searching.
set -uo pipefail
cd "${LAB:?run the setup cell first}" || exit 1

cat > check.py <<'EOF'
import os
import sys

sys.path.insert(0, os.getcwd())          # import from whichever commit git has checked out
try:
    import matrices
    import metrics
except Exception:
    sys.exit(125)                        # 125 tells bisect: cannot judge this commit, skip it
if not hasattr(metrics, "recall"):
    sys.exit(125)

r = metrics.recall(matrices.BALANCED)
print(f"  recall(BALANCED) = {r:.4f}")
sys.exit(0 if abs(r - 0.5) < 1e-9 else 1)    # 0 = good, 1 = bad
EOF

cd analysis || exit 1
GOOD="$(cat "$LAB/GOOD")"                # the commit where recall was first written
echo "known-good commit: $GOOD    known-bad commit: HEAD"
echo

git bisect start
git bisect bad HEAD
git bisect good "$GOOD"
git bisect run "$PY" "$LAB/check.py"
echo
BAD="$(git rev-parse --short refs/bisect/bad)"   # bisect leaves its answer in this ref
git bisect reset
echo
echo "=== and here is what that commit actually did ==="
git show --format='commit %h   %s' "$BAD"

known-good commit: 0d0aa71    known-bad commit: HEAD



status: waiting for both good and bad commits


status: waiting for good commit(s), bad commit known


Bisecting: 4 revisions left to test after this (roughly 2 steps)


[d96dacac465ba6434461b482871c8e77bd87429a] feat(report): show precision next to recall


...

ef _unpack(cm):
+    """Pull the four cells out once so every metric reads the same way."""
+    tn,

 fn = cm[0]
+    fp, tp = cm[1]
+    return tn, fp, fn, tp
+
+
 def accuracy(cm):
-    tn, fp = cm[0

]
-    fn, tp = cm[1]
+    tn, fp, fn, tp = _unpack(cm)
     return (tn + tp) / (tn + fp + fn + tp)


 
 
 def recall(cm):
-    tn, fp = cm[0]
-    fn, tp = cm[1]
+    tn, fp, fn, tp = _unpack(cm)
     

return tp / (tp + fn)


### 👁️ What bisect just told you, and what it did not

Read the output from the bottom. `... is the first bad commit` names **`refactor(metrics): unpack the matrix once instead of in every function`**, and the `git show` under it prints exactly what that commit did: nine lines added, four removed.

Look at the two functions it deleted from — both of them read the matrix as `tn, fp = cm[0]` and `fn, tp = cm[1]`, which is the layout the module docstring on line 1 promises. Now look at the `_unpack` it added:

```python
+def _unpack(cm):
+    """Pull the four cells out once so every metric reads the same way."""
+    tn, fn = cm[0]
+    fp, tp = cm[1]
```

`fp` and `fn` changed places on their way out of the matrix. Three things here are worth more than the bug itself:

1. **It hid on the baseline model.** `BASELINE = [[3191, 3], [3, 3]]` has `fp = 3` and `fn = 3`. Swap two equal numbers and nothing changes. The refactor was, on the data anyone was looking at, invisible. It only shows on `BALANCED`, where `fp = 18` and `fn = 3` — which is precisely the matrix the README makes its claim about.
2. **`accuracy` never broke.** It only ever *adds* all four cells, so their order cannot matter. A test suite that checked accuracy would have passed on every commit in this history.
3. **The commit that broke it is honestly labelled `refactor`.** The author believed nothing was changing. That is not carelessness; that is what a refactor *is*. It is also why `git bisect` beats reading the log: bisect does not care what the message claims.

So the README was right and `metrics.py` was wrong — and you established that without reading `metrics.py` at all.

---

## 6️⃣ Getting out of it

Five mistakes, five different commands. Choosing the wrong one is how a bad hour becomes a bad week, so learn the table, not the commands.

| what you did | what you want | the command | safe to use after pushing? |
|---|---|---|---|
| edited a file, want it back | discard the edit | `git restore <file>` | n/a — nothing was committed |
| `git add`ed the wrong file | unstage it, keep the edit | `git restore --staged <file>` | n/a |
| bad message on the last commit | rewrite it | `git commit --amend` | **no** — rewrites history |
| shipped a commit that is wrong | undo it with a *new* commit | `git revert <sha>` | **yes** — this is the shared-history answer |
| ran a `reset` you regret | go back to where you were | `git reflog` then `git reset --hard HEAD@{n}` | recovers from almost anything |

The rule underneath all five: **`revert` adds history, `reset` and `amend` rewrite it.** Rewriting history that another person has already pulled forces work onto them. On a branch only you have touched, rewrite freely. Once you have pushed it and someone has it, `revert`.

And the safety net: `git reflog` is a log of **everywhere HEAD has been**, including positions no branch points at any more. It is local, it is not pushed, and it keeps entries for 90 days by default. Almost every "I have lost my work" is a reflog away from being solved.

In [10]:
%%bash
# WHAT: the first three rows of the table, run for real: restore, restore --staged, revert.
# WHY:  each one is one command, and the difference between them is which of the three
#       boxes in the picture you are pulling from.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

echo "### 1. discard an unstaged edit --------------------------------"
printf '\nTHIS SENTENCE IS A MISTAKE.\n' >> README.md
git status --short
git restore README.md
git status --short && echo "  (nothing printed: git restore put the file back)"

echo
echo "### 2. unstage a file added by accident ------------------------"
printf 'AWS_SECRET = "this-should-never-be-committed"\n' > credentials.py
git add .
git status --short
git restore --staged credentials.py
echo "  after git restore --staged credentials.py:"
git status --short
echo "  (?? = untracked. The file is still on disk and its contents are untouched;"
echo "   it is simply no longer queued for the next commit.)"
rm credentials.py

echo
echo "### 3. undo a commit that has already been shared --------------"
printf '\nAccuracy is the headline number and should be reported first.\n' >> README.md
git commit -q -am "docs: lead with accuracy"
git revert --no-edit HEAD
echo
git log --oneline -3
echo
echo "  the last two lines of README.md are back to what they were:"
tail -2 README.md

### 1. discard an unstaged edit --------------------------------


 M README.md


  (nothing printed: git restore put the file back)



### 2. unstage a file added by accident ------------------------


A  credentials.py


...

d, 2 deletions(-)


7dcf733 Revert "docs: lead with accuracy"


cd883f8 docs: lead with accuracy
a185b3c feat(metrics): total(), the number of rows a matrix summari

ses

  the last two lines of README.md are back to what they were:



Usage: `python report.py`


In [11]:
%%bash
# WHAT: the accident that frightens people most - `git reset --hard`, which discards commits
#       AND uncommitted work - followed by the recovery.
# WHY:  once you have seen HEAD come back from the reflog you will stop being afraid of git,
#       and being unafraid of git is most of what separates a fast engineer from a slow one.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

echo "=== before ==="
git log --oneline -1
BEFORE="$(git rev-parse --short HEAD)"

echo
echo "=== the accident: git reset --hard HEAD~5 ==="
git reset --hard HEAD~5
git log --oneline -1
echo "  five commits are no longer reachable from main. Nothing was deleted."

echo
echo "=== git reflog: everywhere HEAD has been, most recent first ==="
git reflog -6

echo
echo '=== the recovery: git reset --hard "HEAD@{1}"  (one step before the reset) ==='
git reset --hard "HEAD@{1}"
git log --oneline -1
echo
if [ "$(git rev-parse --short HEAD)" = "$BEFORE" ]; then
  echo "HEAD is back at $BEFORE - byte-for-byte where it was. Nothing was lost."
else
  echo "HEAD is at $(git rev-parse --short HEAD), expected $BEFORE"
fi

=== before ===


7dcf733 Revert "docs: lead with accuracy"


=== the accident: git reset --hard HEAD~5 ===


HEAD is now at 7f00cba style(report): align the assignments


7f00cba style(report): align the assignments


...

b254e to 58dc01c74bf56a5dd508a1297772cf20ddf63b59



=== the recovery: git reset --hard "HEAD@{1}"  (one step before the reset) ===


HEAD is now at 7dcf733 Revert "docs: lead with accuracy"


7dcf733 Revert "docs: lead with accuracy"


HEAD is back at 7dcf733 - byte-for-byte where it was. Nothing was lost.


### The sixth thing that goes wrong: a conflict

A conflict is not an error and not a failure. It is git saying: *two commits changed the same lines and I will not guess which one you meant.* Git edits the file in place, putting both versions in it between markers:

```
<<<<<<< HEAD
THRESHOLD = 0.05          ← what is on the branch you are merging INTO
=======
THRESHOLD = 0.20          ← what is on the branch you are merging FROM
>>>>>>> tune-threshold
```

You resolve it by editing that file until it is the code you want — **markers deleted** — then `git add` it and `git commit`. Or you take the exit: `git merge --abort` puts everything back exactly as it was, every time.

In [12]:
%%bash
# WHAT: cause a real merge conflict on purpose, look at what git writes into the file, and
#       then take the exit.
# WHY:  the first conflict you meet should not be one you are under pressure to resolve.
set -uo pipefail
cd "${LAB:?run the setup cell first}" || exit 1

rm -rf conflict-demo && mkdir conflict-demo && cd conflict-demo || exit 1
git init -q -b main
git config user.name "Student" && git config user.email "student@example.com"
printf 'THRESHOLD = 0.10\n' > config.py
git add config.py && git commit -q -m "chore: set the alert threshold to 0.10"

git switch -q -c tune-threshold
printf 'THRESHOLD = 0.20\n' > config.py
git commit -q -am "tune: raise the cut to 0.20 - same fraud caught, half the false alarms"

git switch -q main
printf 'THRESHOLD = 0.05\n' > config.py
git commit -q -am "tune: lower the cut to 0.05 to catch more fraud"

echo "=== git merge tune-threshold ==="
git merge tune-threshold
echo
echo "=== what git wrote into config.py ==="
cat config.py
echo
echo "=== git status --short   (UU = unmerged, both sides changed it) ==="
git status --short
echo
echo "=== the exit ==="
git merge --abort
echo "config.py after git merge --abort:"
cat config.py
git status --short && echo "(clean - as if the merge had never been attempted)"

=== git merge tune-threshold ===


Auto-merging config.py
CONFLICT (content): Merge conflict in config.py


Automatic merge failed; fix conflicts and then commit the result.



=== what git wrote into config.py ===


<<<<<<< HEAD
THRESHOLD = 0.05
THRESHOLD = 0.20
>>>>>>> tune-threshold


...

UU config.py



=== the exit ===


config.py after git merge --abort:


THRESHOLD = 0.05


(clean - as if the merge had never been attempted)


---

## 👥 The second half: working with another person

Everything above works alone. None of it is why git exists.

### What a "pull request" actually is

A pull request is not a GitHub feature. Strip the web page away and it is three things:

1. **A branch** containing your commits.
2. **A diff** between that branch and the branch you want it to join — the thing to be reviewed.
3. **A person other than you** who reads that diff and decides.

Git had this before the web page did: `git request-pull` literally writes the message asking someone to pull your branch. Everything below runs on your own machine with **no network, no GitHub account, and no sign-up** — because a git "remote" can simply be another directory.

That matters for more than convenience. It means the loop is: *branch → commit → push → someone reads the diff → merge*, and the web interface is only paint on top of it. Learn the loop and every hosting service is a detail.

**The four commands of the loop:**

| step | command | who runs it |
|---|---|---|
| get a copy | `git clone <where>` | the person joining |
| propose | `git switch -c <name>` … `git push origin <name>` | the author |
| review | `git log main..<name>` and `git diff main...<name>` | the reviewer |
| accept | `git merge --no-ff <name>` | the maintainer |

Note the diff has **three** dots. `git diff main..branch` compares the two tips. `git diff main...branch` compares the branch against **the point where it left main** — which is what the author actually changed, with everyone else's later work on main excluded. Three dots is what a pull request shows you, and it is what you want.

In [13]:
%%bash
# WHAT: a second person clones the project, fixes the bug bisect found, and pushes the fix
#       back as a branch. "origin" here is just a path on this machine - no network at all.
# WHY:  this is the entire collaboration loop, with nothing hidden behind a web page.
set -uo pipefail
cd "${LAB:?run the setup cell first}" || exit 1

echo "=== a colleague gets a copy ==="
rm -rf colleague
git clone "$LAB/analysis" "$LAB/colleague"
cd colleague || exit 1
git config user.name "Sara"                    # a different person, so the history says so
git config user.email "sara@example.com"
git config commit.gpgsign false

echo
echo "=== she works on a branch, never on main ==="
git switch -c fix-recall

"$PY" - <<'EOF'
# The fix itself: put fp and fn back in the order the docstring promises.
import pathlib
p = pathlib.Path("metrics.py")
s = p.read_text()
s = s.replace("    tn, fn = cm[0]\n    fp, tp = cm[1]\n",
              "    tn, fp = cm[0]\n    fn, tp = cm[1]\n")
p.write_text(s)
EOF

git commit -q -am "fix(metrics): _unpack returned fp and fn swapped, so recall was precision

The refactor that introduced _unpack read row 0 as (tn, fn) and row 1 as
(fp, tp); the layout in the module docstring, and everywhere else, is
[[tn, fp], [fn, tp]].

On BASELINE the bug is invisible because fp == fn == 3. On BALANCED it made
recall report 0.1429 instead of 0.5000 - which is precision - and so
contradicted the README's claim that class_weight left recall untouched.
accuracy was never affected: it only sums the four cells.

Found with git bisect run against a one-assertion check on recall(BALANCED)."

echo
echo "=== she pushes the BRANCH (not main) back to origin ==="
git push origin fix-recall

=== a colleague gets a copy ===


Cloning into '/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/git-lab-c955n6hk/colleague'...


done.



=== she works on a branch, never on main ===


Switched to a new branch 'fix-recall'



=== she pushes the BRANCH (not main) back to origin ===


To /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/git-lab-c955n6hk/analysis
 * [new branch]      f

ix-recall -> fix-recall


### Now the review — the part that is actually the job

The maintainer's side of a pull request is two commands: read the list of commits, read the combined diff. Everything else is judgement.

In [14]:
%%bash
# WHAT: everything the reviewer sees, from the maintainer's own clone.
# WHY:  these three commands ARE a pull request page. Learn them and no hosting service can
#       confuse you.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

echo "=== 1. what commits are being proposed?   git log main..fix-recall ==="
git log --oneline main..fix-recall
echo
echo "=== 2. what do they change?   git diff main...fix-recall ==="
git diff main...fix-recall
echo
echo "=== 3. the original pull request, before the web page existed ==="
git request-pull main "$LAB/colleague" fix-recall

=== 1. what commits are being proposed?   git log main..fix-recall ===


16636ae fix(metrics): _unpack returned fp and fn swapped, so recall was precision



=== 2. what do they change?   git diff main...fix-recall ===


diff --git a/metrics.py b/metrics.py
index c3542ec..cba4bad 100644
--- a/metrics.py
+++ b/metrics.py


@@ -3,8 +3,8 @@
 
 def _unpack(cm):
     """Pull the four cells out once so every metric reads the 

...


for you to fetch changes up to 16636ae1dd39278fa60fdbab8b9d738eaa636f43:

  fix(metrics): _unpack r

eturned fp and fn swapped, so recall was precision (2026-09-05 00:51:11 +0300)

--------------------

--------------------------------------------


Sara (1):
      fix(metrics): _unpack returned fp and fn swapped, so recall was precision



 metrics.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


### How to write the review

Reviewing is a skill with an evidence base, and the evidence is unusually specific about *what kind* of comment helps.

Wisniewski, Zierer & Hattie (2020) meta-analysed 435 studies, 994 effect sizes, and more than 61,000 people. Feedback carrying **high information about the task and the process** had an effect of **d = 0.99** (95% CI 0.82–1.15). Merely **corrective** feedback managed **d = 0.46**. Reinforcement or punishment reached **d = 0.24**, and **praise was the least effective category in the analysis.** Stated openly: John Hattie is one of the authors of the 2007 framework the paper evaluates, which is a conflict of interest you should weigh.

The practical consequence is not "give more feedback". It is: **"nice work" is not a review, and neither is "looks good to me".** So the form below is built so that you *cannot* produce a praise-only or person-directed review and still have filled it in — every field demands a file, a line, or something you ran.

> **Code review form — every field is required**
>
> 1. **In one sentence, in your own words, what does this change do?** (If you cannot write this, you have not read it yet.)
> 2. **What did you run, and what did it print?** Paste the actual output. Not "seems fine".
> 3. **One thing that is wrong, or will be wrong later.** Give `file:line`. If you genuinely find nothing, write what you *checked* and why it convinced you.
> 4. **One thing you would do differently, and why.** About the code or the approach — never about the author.
> 5. **Decision:** approve / approve with changes / needs another round.
>
> **Two rules.** No comment may name the author or use "you". Every comment must name a file and a line. "This variable name is misleading" is a review; "you always name things badly" is not.

**A worked example against the branch above:**

- *(1)* Restores the `(tn, fp)` / `(fn, tp)` unpacking order in `_unpack`, so `recall` computes recall again.
- *(2)* Ran `check.py` against the merged result: it printed `recall(BALANCED) = 0.5000` and exited 0. The same check on main before the fix printed `0.1429`.
- *(3)* `metrics.py:6` — the fix is correct but nothing stops the next refactor from re-swapping them. There is still no test in the repository; `check.py` lives outside it.
- *(4)* I would have `_unpack` return a small named structure rather than a four-tuple, because positional tuples are exactly what made this bug possible. Follow-up commit, not a blocker.
- *(5)* Approve — with the test tracked as follow-up work.

Notice that comment (3) is worth more than the fix. It found the *class* of defect, not the instance. That is what a reviewer is for, and it is the reason review beats testing at some things and loses to it at others.

In [15]:
%%bash
# WHAT: accept the branch with a merge commit, then prove the defect is actually gone by
#       re-running the same check that found it.
# WHY:  --no-ff forces a merge commit even when git could fast-forward, so the history keeps
#       a permanent record that these commits arrived together, as a reviewed unit.
set -uo pipefail
cd "${LAB:?run the setup cell first}/analysis" || exit 1

git merge --no-ff fix-recall \
  -m "merge: fix-recall - _unpack swapped fp and fn (reviewed: metrics.py:6, test still owed)"
echo
echo "=== the shape of the history now ==="
git log --graph --oneline -6
echo
echo "=== the line the review named, with its number ==="
grep -n "tn, fp = cm\[0\]" metrics.py
echo
echo "=== the same check that failed before the merge ==="
"$PY" "$LAB/check.py" && echo "  exit 0 - good"
echo
echo "=== and the report the whole lesson started from ==="
"$PY" report.py

Merge made by the 'ort' strategy.


 metrics.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)



=== the shape of the history now ===


*   0055be0 merge: fix-recall - _unpack swapped fp and fn (reviewed: metrics.py:6, test still owed)


|\  


...

  recall(BALANCED) = 0.5000


  exit 0 - good



=== and the report the whole lesson started from ===


BASELINE  n=3200  accuracy=0.9981  recall=0.5000  precision=0.5000
BALANCED  n=3200  accuracy=0.9934

  recall=0.5000  precision=0.1429


### 👁️ Read the last two lines

`BALANCED` now reports `recall=0.5000` and `precision=0.1429` — the two numbers have swapped back into the right columns. `BASELINE` is unchanged at `recall=0.5000`, `precision=0.5000`, exactly as predicted: on that matrix `fp == fn`, so the bug never showed there and the fix cannot show there either.

And the README's claim — *class_weight moves false positives from 3 to 18 and leaves recall exactly where it was* — is now true of the code as well as of the world. That sentence was the only thing in the whole repository that noticed something was wrong.

---

## 🛠️ Your turn

**Do this in a terminal, not in this notebook.** The notebook has shown you every command; typing them yourself is the part that lasts. Nothing below touches this repository — you build your own from nothing.

```bash
mkdir -p ~/git-practice && cd ~/git-practice
git init -b main
git config user.name "Your Name"
git config user.email "you@example.com"
```

**Task 1 — deliberate staging (this is the one people skip; do not skip it).**
Create one file with two unrelated changes in it — say a function you want to keep and a `print()` you do not. Run `git add -p`, and use `y` and `n` to stage only the first hunk. Then `git diff --staged` and `git diff`.
*Done when:* `git diff --staged` shows the function and `git diff` shows the print.

**Task 2 — a message that survives.**
Commit it with `git commit` (no `-m`, so your editor opens). Write a subject line and a body. Then, before you look at it again, write down on paper the question you think a stranger would ask about this commit. Run `git log -1` and check whether your message answers it.
*Done when:* it does — or you have run `git commit --amend` until it does.

**Task 3 — break something on purpose, then find it.**
Make at least eight commits. Somewhere in the middle, change a number so that a small script prints the wrong answer. Write a `check.py` **outside** the repository that exits 0 when the answer is right and 1 when it is wrong. Then `git bisect start`, `git bisect bad HEAD`, `git bisect good <an early commit>`, `git bisect run python3 ../check.py`.
*Done when:* git names the commit you actually broke it in. If it names a different one, your check is testing the wrong thing — that is a real and common failure, and worth the ten minutes.

**Task 4 — the accident and the recovery.**
Run `git reset --hard HEAD~3`. Confirm with `git log --oneline` that three commits are gone. Get them back.
*Done when:* `git log --oneline` matches what it showed before the reset, and you did it from `git reflog` rather than from memory.

**Task 5 — two people, one machine.**
`git clone ~/git-practice ~/git-practice-partner`. In the clone, make a branch, fix the bug from Task 3, push the branch back. In the original, review it with `git log main..<branch>` and `git diff main...<branch>`, fill in the five-field review form **in writing**, then `git merge --no-ff`.
*Done when:* `git log --graph --oneline` shows a merge commit with two parents, and you have a written review with a `file:line` in field 3.

**Task 6 — swap.** Give your `~/git-practice` to another student. Ask them for theirs. Review each other's Task 3 bug using the same form. Compare: did they find the bug faster from your commit messages than you found theirs?

When you are finished, `rm -rf ~/git-practice ~/git-practice-partner`.

In [16]:
# WHAT: delete the laboratory directory this notebook created.
# WHY:  it lives in the system temp area and nothing else refers to it. Leaving scratch
#       repositories lying around is how people later merge the wrong one.
import shutil
from pathlib import Path

lab = Path(LAB)
n_files = sum(1 for _ in lab.rglob("*")) if lab.exists() else 0
shutil.rmtree(lab, ignore_errors=True)
print(f"removed {LAB} ({n_files} entries)")
print("still exists:", lab.exists())
print("this repository was never written to by this notebook - only read.")

removed /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/git-lab-c955n6hk (355 entries)
still exists: False
this repository was never written to by this notebook - only read.


---

## ✅ Summary

### The twenty commands, in the order you will need them

| doing | command |
|---|---|
| what is dirty | `git status --short` |
| what did I change | `git diff` |
| what am I about to commit | `git diff --staged` |
| stage a file / a hunk | `git add <path>` / `git add -p` |
| unstage | `git restore --staged <path>` |
| discard an edit | `git restore <path>` |
| record it | `git commit -m "subject" -m "body"` |
| fix the last message | `git commit --amend` *(unpushed only)* |
| new branch | `git switch -c <name>` |
| back to the previous branch | `git switch -` |
| delete a branch | `git branch -D <name>` |
| subject lines | `git log --oneline` |
| which commit added this string | `git log -S"<text>" -- <path>` |
| who last touched this line | `git blame <file>` |
| which commit changed the behaviour | `git bisect start` / `bad` / `good` / `run <cmd>` |
| undo a shipped commit | `git revert <sha>` |
| where has HEAD been | `git reflog` |
| get back to where I was | `git reset --hard "HEAD@{n}"` |
| get a copy | `git clone <path-or-url>` |
| propose | `git push origin <branch>` |
| review | `git log main..<b>` and `git diff main...<b>` |
| accept | `git merge --no-ff <branch>` |
| escape a conflict | `git merge --abort` |

### The four ideas underneath them

1. **Three places, not one.** Working tree, index, HEAD. `git status` tells you which two disagree; `git diff` and `git diff --staged` are two different questions.
2. **A commit message is evidence, not decoration.** It is the only place the *reason* can live, and it is read by strangers under time pressure.
3. **`revert` adds history; `reset` and `amend` rewrite it.** Rewrite only what nobody else has. `git reflog` forgives almost everything else.
4. **A pull request is a branch, a diff, and a second pair of eyes.** The web page is optional; the second pair of eyes is not.

### What you did in this notebook

Built a twelve-commit repository; read a working tree three ways; staged one file and left another; wrote a message with a subject and a body; branched, worked, and abandoned the branch; used the pickaxe on this repository's real history to reconstruct why four lines of code exist; used `git bisect run` to identify a defect-introducing commit in four automated checks without reading the broken function; discarded an edit, unstaged a file, reverted a commit, recovered from `git reset --hard` via the reflog, and walked into a merge conflict and back out of it; then cloned, branched, pushed, reviewed and merged — offline, with no account anywhere.

## 💬 Discuss

1. The bug survived seven later commits because `BASELINE` had `fp = fn = 3`. Think of a dataset you have used in this diploma where a swapped pair of values would be **equally invisible**. What does that tell you about choosing test data — and about why "the tests pass" is a weaker statement than it sounds?
2. `git revert` leaves the mistake in the history forever; `git reset` makes it disappear. A manager asks you to `reset` a commit that has already been pushed, because it is "embarrassing". Give the technical argument against, and then the argument that is really about trust. Which one do you make out loud?
3. The review form forbids the word "you". One student says that is patronising, another says it is the only reason the reviews stay useable after week three. Argue both, then say what you would do in a class where everyone knows each other well.
4. `git bisect` needs a test that exits 0 or 1. For a machine-learning defect — a model whose accuracy quietly dropped from 0.91 to 0.87 — writing that test is much harder than it was here. What would you assert, and what would you do about run-to-run variation? (There is no clean answer. Say what you would actually type.)
5. Everything in the second half of this lesson ran with no network and no account. Who benefits from that being possible, and what would you lose if the only way you knew how to collaborate was through one company's web interface?

## ⚠️ Where this breaks

**Start with the honest one: this whole tooling strand has no outcome evidence behind it.**
There is no study we can point you to showing that teaching git makes AI students better at AI. It is in this programme on a *prerequisite* argument, not a proven one: peer review on real work, written feedback inside artefacts, and being handed a repository at all are impossible without it. That is an argument from necessity. Weigh it as such, and do not let anyone tell you it is more.

The peer-review half rests on firmer ground, with limits that must travel with the number. Double, McGrane & Hopfenbeck (2020) found peer assessment improved university students' performance at **g = 0.31** overall (95% CI 0.18–0.44) — slightly ahead of teacher assessment at **g = 0.28** — and **g = 0.55** where the peer feedback carried a grade. But: the effect **could not be shown at all for primary or secondary students** (g = 0.002, not significant); the corpus is largely quasi-experimental with **whole classes** assigned rather than individuals; **geography was never tested as a moderator**, so there is *no evidence either way* about how this transfers to Saudi learners; and almost none of the 54 studies involved code. You are, in a small way, outside the evidence.

**Where the git content itself breaks:**

- **`git bisect` needs a deterministic, fast test.** Ours ran in milliseconds and gave the same answer every time. Bisecting a defect that appears in one run out of five will confidently name an innocent commit. Bisecting something that takes forty minutes to reproduce is a day's work, not four checks. *Then use:* `git bisect skip` for untestable commits, and be ruthless about shrinking the reproduction first.
- **`git bisect` assumes the behaviour changed exactly once.** Two bugs — or a bug introduced, fixed, and reintroduced — break the halving argument, and bisect will still report a single confident answer.
- **The pickaxe (`-S`) only finds strings that appear in a diff.** A defect caused by a *deleted* condition, a changed dependency version, or a data file will not be found this way at all.
- **`git blame` names the last person to touch a line, not the person who caused the problem.** A whitespace reformat makes every line yours. `git blame -w` ignores whitespace and `--ignore-rev` skips a known reformat commit; use them before you accuse anyone of anything.
- **`git reflog` is local, unpushed, and expires** (90 days by default, 30 for unreachable commits). It cannot recover work from a repository you deleted, and it does not exist in a fresh clone.
- **`git restore <file>` is genuinely destructive.** Uncommitted work has never been in git and cannot be recovered by it. This is the one command in the lesson with no undo.
- **Merging is not the only model.** Teams that require a linear history use `git rebase` instead, which rewrites commits and carries its own failure modes. We taught `merge` because it never rewrites anything and is therefore the safe default for someone who has been using git for two hours.
- **Git is bad at large binary files and at notebooks.** A `.ipynb` is JSON containing outputs, so two people running the same notebook produce a large conflicting diff of nothing. Teams handle this with `nbstripout`, `jupytext`, or a review rule that only source cells count. Expect it; it is not your mistake.
- **The assumption that must hold for the second half:** that a second person will actually read the diff. A review that rubber-stamps is worse than no review, because it manufactures a record of scrutiny that did not happen. Bacchelli & Bird's study of code review at Microsoft found that what teams *expect* from review (finding defects) and what they mostly *get* (understanding the code, spreading knowledge) are not the same thing — which is an argument for reviewing, and against believing review is your defect net.

## 📚 References

1. Chacon, S., & Straub, B. (2014). *Pro Git* (2nd ed.). Apress. — The reference text; chapters 2, 3 and 7 cover everything in this lesson and are the standard answer to "where do I look it up".
2. Torvalds, L., Hamano, J. C., et al. (2005–). *Git*. The manual pages shipped with your installation: `git help log`, `git help bisect`, `git help revert`. `git help -g` lists the guides, including `gitworkflows` and `giteveryday`.
3. Wisniewski, B., Zierer, K., & Hattie, J. (2020). The Power of Feedback Revisited: A Meta-Analysis of Educational Feedback Research. *Frontiers in Psychology*, 10:3087. — 435 studies, 994 effect sizes, N > 61,000; task-and-process feedback d = 0.99 [0.82–1.15], corrective d = 0.46, reinforcement/punishment d = 0.24, praise least effective. Conflict of interest: Hattie co-authored the 2007 framework the paper evaluates.
4. Double, K. S., McGrane, J. A., & Hopfenbeck, T. N. (2020). The Impact of Peer Assessment on Academic Performance: A Meta-analysis of Control Group Studies. *Educational Psychology Review*, 32, 481–509. — 54 studies; overall g = 0.31 (CI 0.18–0.44); g = 0.55 for graded peer feedback at university level; g = 0.002 (n.s.) for primary and secondary students.
5. Bacchelli, A., & Bird, C. (2013). Expectations, Outcomes, and Challenges of Modern Code Review. *Proceedings of the 35th International Conference on Software Engineering (ICSE '13)*. — Interviews and surveys at Microsoft; the outcomes teams expect from review and the outcomes they actually obtain differ, with knowledge transfer outweighing defect detection.
6. This repository's own history — commits `a1249287`, `79309823`, `5113c385` and `ced3fad6`, quoted in the cells above. Read them yourself with `git show <sha>`; that is the only citation in this list you can verify without leaving your machine.

---

*Every number quoted in this notebook's prose is either printed by a cell above it or comes from the source named beside it. If you find one that is not, that is a defect — and you now know how to write the commit message that fixes it.*